In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_10.pickle

trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['bar_chart_multiple_choice_multiple_selection']
me:  2
trying: ['count_then_return_percent_17']
me:  16
trying: ['create_choropleth_map']
me:  2
trying: ['bar_chart_multiple_choice']
me:  2
trying: ['bar_chart_multiple_years']
me:  2
trying: ['convert_df_of_counts_to_percentages']
me:  4
trying: ['create_dataframe_of_counts']
me:  4
trying: ['count_then_return_percent']
me:  4
trying: ['orig_output']
me:  15
trying: ['add_year_column_to_dataframes']
me:  4
trying: ['glob']
me:  0
trying: ['title_for_y_axis']
me:  20
trying: ['grab_subset_of_data']
me:  4
trying: ['add_year_column_to_dataframes_18']
me:  18
trying: ['load_survey_data']
me:  6
trying: ['warnings']
me:  0
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  18
trying: ['sns']
me:  0
trying: ['responses_df_2022']


me:  18
trying: ['base_dir_2021']
me:  8
trying: ['percentages']
me:  20
trying: ['bar_chart_multiple_years_18']
me:  18
trying: ['file_name_2021']
me:  8
trying: ['title_for_x_axis']
me:  18
trying: ['base_dir_2019']
me:  8
trying: ['question_name_alternate']
me:  18
trying: ['count_then_return_percent_19']
me:  20
trying: ['count_then_return_percent_18']
me:  18
trying: ['base_dir_2018']
me:  8
trying: ['bar_chart_multiple_choice_19']
me:  20
trying: ['country_df_combined']
me:  18
trying: ['question_of_interest']
me:  20
trying: ['subset_of_countries']
me:  18
trying: ['responses_df_2020']


me:  8
trying: ['responses_df_2017']


me:  18
trying: ['responses_df_2021']


me:  8
trying: ['responses_df_2019']


me:  10
trying: ['column_of_interest']
me:  18
trying: ['question_name']
me:  18
trying: ['responses_per_country_df']
me:  14
trying: ['create_choropleth_map_16']
me:  14
trying: ['percentages_per_country_df']
me:  14
trying: ['base_dir_2020']
me:  8
trying: ['base_dir_2017']
me:  8
trying: ['directory']
me:  8
trying: ['base_dir_2022']
me:  8
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions']
me:  6
trying: ['create_dataframe_of_counts_16']
me:  14
trying: ['combine_subset_of_data_from_multiple_years']
me:  6
trying: ['file_name_2022']
me:  8
trying: ['label_for_legend']
me:  14
trying: ['px']
me:  0
trying: ['file_name_2019']
me:  8
trying: ['go']
me:  0
trying: ['file_name_2020']
me:  8
trying: ['orientation_for_chart']
me:  16
trying: ['title_for_chart']
me:  20
trying: ['replace_hyphen_with_en_dash']
me:  10
trying: ['bar_chart_multiple_choice_17']
me:  16
trying: ['responses_df_2018']


me:  10
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['responses_in_order']
me:  16
Declaring variable glob
Declaring variable warnings
Declaring variable sns
Declaring variable px
Declaring variable go
Declaring variable np
Declaring variable pd
Declaring variable bar_chart_multiple_choice_multiple_selection
Declaring variable create_choropleth_map
Declaring variable bar_chart_multiple_choice
Declaring variable bar_chart_multiple_years
Declaring variable convert_df_of_counts_to_percentages
Declaring variable create_dataframe_of_counts
Declaring variable count_then_return_percent
Declaring variable add_year_column_to_dataframes
Declaring variable grab_subset_of_data
Declaring variable load_survey_data
Declaring variable combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions
Declaring variable combine_subset_of_data_from_multiple_years
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declarin

In [4]:
%%RecordEvent
%%time
### cell 11 ###

# Existing utility: simply export the combined DataFrame to CSV
def bar_chart_multiple_years_20(df, x_column, y_column, title, y_axis_title):
    df.to_csv(
        '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'
        + title + '.csv',
        index=True
    )


def count_then_return_percent_20(dataframe, x_axis_title):
    # Vectorized count and percent calculation (runs on GPU via cudf.pandas)
    col = dataframe[x_axis_title]
    counts = col.value_counts(dropna=False)
    total = col.count()
    percentages = (counts * 100 / total).round(1)
    return percentages


def combine_subset_of_data_from_multiple_years_20(
        question_of_interest,
        x_axis_title,
        y_column='percentage',
        include_2017=None
    ):
    # Prepare list of years and corresponding DataFrames
    year_dfs = [
        ('2018', responses_df_2018),
        ('2019', responses_df_2019),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022),
    ]
    if include_2017 is not None:
        year_dfs.insert(0, ('2017', responses_df_2017))

    dfs = []
    for year, df in year_dfs:
        # compute percentages series and sort
        perc = count_then_return_percent_20(df, question_of_interest).sort_index()
        # build a per‐year DataFrame using the y_column variable (not a hard‐coded string)
        df_year = pd.DataFrame({y_column: perc.values})
        df_year['year'] = year
        df_year[x_axis_title] = perc.index
        dfs.append(df_year)

    # concatenate all years together
    df_combined = pd.concat(dfs, ignore_index=True)
    return df_combined

# --- bottom of cell ---
question_of_interest = 'What is your age (# years)?'
column_of_interest = 'percentage'
title_for_chart = 'Age distributions on Kaggle (2018-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'

# Merge the top age brackets into '70+'
responses_df_2018[question_of_interest].replace(['70-79', '80+'], '70+', inplace=True)

# Build and export the chart, passing column_of_interest into the combine function
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest,
    title_for_x_axis,
    column_of_interest
)
bar_chart_multiple_years_20(
    age_df_combined,
    title_for_x_axis,
    column_of_interest,
    title_for_chart,
    title_for_y_axis
)

CPU times: user 290 ms, sys: 27.6 ms, total: 318 ms
Wall time: 318 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_11_try_7.pickle

migration speed (bps): 789362245.3400301
---------------------------
variables to migrate:
question_of_interest 76
title_for_y_axis 65
orig_output 16
bar_chart_multiple_years_20 144
title_for_chart 88
create_dataframe_of_counts_16 144
label_for_legend 65
responses_per_country_df 48
pd 72
create_choropleth_map_16 144
percentages_per_country_df 48
bar_chart_multiple_choice 144
bar_chart_multiple_choice_multiple_selection 144
responses_df_2017 48
bar_chart_multiple_years 144
base_dir_2020 155
add_year_column_to_dataframes_18 144
combine_subset_of_data_from_multiple_years_20 144
base_dir_2017 155
combine_subset_of_data_from_multiple_years_18 144
create_choropleth_map 144
directory 170
column_of_interest 59
age_df_combined 48
base_dir_2022 155
question_name 90
count_then_return_percent_20 144
bar_chart_multiple_years_18 144
file_name_2022 81
title_for_x_axis 49
file_name_2018 76
file_name_2019 78
question_name_alternate 70
file_name_2020 81
count_then_return_percent_18 144
country_df_combin

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'responses_df_2022']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'responses_df_2022']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'responses_df_2022']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['responses_df_2019', 'responses_df_2018', 'responses_df_2017', 'responses_df_2022', 'responses_df_2021', 'responses_df_2020']
Intermediate variables ['base_dir_2019', 'base_dir_2022', 'base_dir_2017', 'base_dir_2020', 'base_dir_2018', 'file_name_2020', 'file_name_2017', 'file_name_2019', 'base_dir_2021', 'directory', 'file_n

In [7]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_11_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[11], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
assert False, 'question_of_interest is incorrectly modified in the optimized code.'